In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [2]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [3]:
from sklearn.ensemble import RandomForestClassifier

In [56]:
from sklearn.metrics import accuracy_score

In [57]:
from xgboost import XGBClassifier

In [58]:
from lightgbm import LGBMClassifier

In [4]:
from sklearn.preprocessing import LabelEncoder

In [6]:
train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')

In [7]:
test=pd.read_csv("/kaggle/input/competitions/spaceship-titanic/test.csv")

In [8]:
print(train.shape, test.shape)

(8693, 14) (4277, 13)


In [28]:
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,VRDeck,Name,Transported,Deck,CabinNum,Side,Group,GroupSize,TotalSpend,NoSpend
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,...,0.0,Maham Ofracculy,False,B,0.0,P,0001,1,0.0,1
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,...,44.0,Juanna Vines,True,F,0.0,S,0002,1,736.0,0
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,...,49.0,Altark Susent,False,A,0.0,S,0003,2,10383.0,0
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,...,193.0,Solam Susent,False,A,0.0,S,0003,2,5176.0,0
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,...,2.0,Willy Santantines,True,F,1.0,S,0004,1,1091.0,0


In [59]:
def engineer_features(df):
    df = df.copy()
 
    df[['Deck', 'CabinNum', 'Side']] = df['Cabin'].str.split('/', expand=True)
    df['CabinNum'] = pd.to_numeric(df['CabinNum'], errors='coerce')
 
    df['Group'] = df['PassengerId'].str.split('_').str[0]
    group_sizes = df['Group'].value_counts()
    df['GroupSize'] = df['Group'].map(group_sizes)
    df['IsAlone'] = (df['GroupSize'] == 1).astype(int)
 
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df['TotalSpend'] = df[spend_cols].sum(axis=1)
    df['NoSpend'] = (df['TotalSpend'] == 0).astype(int)
    df['LogTotalSpend'] = np.log1p(df['TotalSpend'])
 
    # Name -> Surname (people traveling together often share a surname)
    df['Surname'] = df['Name'].str.split().str[-1]
    surname_counts = df['Surname'].value_counts()
    df['SurnameGroupSize'] = df['Surname'].map(surname_counts)
 
    # Age buckets - kids and elderly can behave differently
    df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100],
                             labels=['Child', 'Teen', 'Adult', 'MidAge', 'Senior'])
 
    df['CryoSleep'] = df['CryoSleep'].fillna(False)
 
    return df
 
train = engineer_features(train)
test = engineer_features(test)

In [60]:
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,Side,Group,GroupSize,TotalSpend,NoSpend,IsAlone,LogTotalSpend,Surname,SurnameGroupSize,AgeGroup
0,0001_01,1,0,B/0/P,2,39.0,0,0.0,0.0,0.0,...,P,0001,1,0.0,1,1,0.000000,Ofracculy,1.0,MidAge
1,0002_01,0,0,F/0/S,2,24.0,0,109.0,9.0,25.0,...,S,0002,1,736.0,0,1,6.602588,Vines,4.0,Adult
2,0003_01,1,0,A/0/S,2,58.0,1,43.0,3576.0,0.0,...,S,0003,2,10383.0,0,0,9.248021,Susent,6.0,MidAge
3,0003_02,1,0,A/0/S,2,33.0,0,0.0,1283.0,371.0,...,S,0003,2,5176.0,0,0,8.551981,Susent,6.0,Adult
4,0004_01,0,0,F/1/S,2,16.0,0,303.0,70.0,151.0,...,S,0004,1,1091.0,0,1,6.995766,Santantines,6.0,Teen


In [31]:
test.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Deck,CabinNum,Side,Group,GroupSize,TotalSpend,NoSpend
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning,G,3.0,S,0013,1,0.0,1
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers,F,4.0,S,0018,1,2832.0,0
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus,C,0.0,S,0019,1,0.0,1
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter,C,1.0,S,0021,1,7418.0,0
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez,F,5.0,S,0023,1,645.0,0


In [61]:
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
            'CabinNum', 'TotalSpend', 'LogTotalSpend', 'GroupSize', 'SurnameGroupSize']

In [62]:
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side', 'AgeGroup']

In [63]:
for col in num_cols:
    median_val = train[col].median()
    train[col] = train[col].fillna(median_val)
    test[col] = test[col].fillna(median_val)

In [64]:
for col in cat_cols:
    mode_val = train[col].mode()[0]
    train[col] = train[col].fillna(mode_val)
    test[col] = test[col].fillna(mode_val)

In [65]:
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col].astype(str), test[col].astype(str)])
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))
 

In [67]:
features = num_cols + cat_cols + ['NoSpend', 'IsAlone']

In [68]:
X = train[features]

In [69]:
y = train['Transported'].astype(int)

In [70]:
X_test = test[features]

In [71]:
rf_model = RandomForestClassifier(
    n_estimators=500, max_depth=12, min_samples_split=5,
    random_state=42, n_jobs=-1
)

In [72]:
xgb_model = XGBClassifier(
    n_estimators=600, max_depth=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='logloss', n_jobs=-1
)
 

In [74]:
lgbm_model = LGBMClassifier(
    n_estimators=600, max_depth=6, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbose=-1
)
 

In [75]:
models = {'RandomForest': rf_model, 'XGBoost': xgb_model, 'LightGBM': lgbm_model}

In [76]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [77]:
oof_preds = {name: np.zeros(len(X)) for name in models}

In [78]:
test_preds = {name: np.zeros(len(X_test)) for name in models}

In [79]:
for name, model in models.items():
    fold_scores = []
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
 
        model.fit(X_tr, y_tr)
        val_pred = model.predict_proba(X_val)[:, 1]
        oof_preds[name][val_idx] = val_pred
        test_preds[name] += model.predict_proba(X_test)[:, 1] / cv.get_n_splits()
 
        fold_acc = accuracy_score(y_val, val_pred > 0.5)
        fold_scores.append(fold_acc)
 
    print(f"{name}: CV Accuracy = {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")
 

RandomForest: CV Accuracy = 0.8001 (+/- 0.0085)
XGBoost: CV Accuracy = 0.8094 (+/- 0.0061)
LightGBM: CV Accuracy = 0.8097 (+/- 0.0065)


In [52]:
submission.to_csv('submission.csv', index=False)

In [80]:
ensemble_oof = np.mean([oof_preds[name] for name in models], axis=0)

In [81]:
ensemble_acc = accuracy_score(y, ensemble_oof > 0.5)

In [82]:
print(f"\nEnsemble CV Accuracy: {ensemble_acc:.4f}")


Ensemble CV Accuracy: 0.8095


In [83]:
ensemble_test = np.mean([test_preds[name] for name in models], axis=0)

In [84]:
final_predictions = (ensemble_test > 0.5)

In [85]:
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': final_predictions
})

In [86]:
submission.to_csv('submission.csv', index=False)
print("\nsubmission.csv created — ready to submit!")


submission.csv created — ready to submit!
